In [1]:
import torch
import torch.nn as nn

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0 ), \
        "d_out must be divisible by num_heads (number of heads)"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out) # Linear layer to combine head output
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
    def forward(self,x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)      # Shape:(b, num_tokens, d_out)
        queries = self.W_query(x) # Shape:(b, num_tokens, d_out)
        values = self.W_value(x)  # Shape:(b, num_tokens, d_out)

    # We implicitly split the matrix by adding a 'num_heads' dimension
    # Unroll last dim: (b, num_tokens, d_out) --> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

    # Transpose: (b, num_tokens, num_heads, head_dim) --> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

    # Compute scaled dot-product attention (aka self-attention) with a causal mask
        atten_scores = queries @ keys.transpose(2,3) # Dot-Product for each head

    # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

    # Use the mask to fill attention scores
        atten_scores.masked_fill_(mask_bool, -torch.inf)

    # Find the attention weights 
        atten_weight = torch.softmax(atten_scores / keys.shape[-1] ** 0.5, dim=-1)
        atten_weight = self.dropout(atten_weight)

    # Shape: (b, num_tokens, num_heads, head_dim)
        context_vector = (atten_weight @ values).transpose(1,2)

    # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vector = context_vector.contiguous().view(b, num_tokens, self.d_out)
        context_vector = self.out_proj(context_vector) # optional projection

        return context_vector
    

NameError: name 'nn' is not defined